# MicroDuck · 官方步态交互遥控
这份 Notebook 使用和 Qwen 演示相同的 2×2 MuJoCo 场景与 Pollen Robotics 的官方 ONNX 步态，**无需下载 Qwen 权重，也不需要 GPU**。

按顺序运行前两格后，第 3 格会启动常驻的交互遥控台：你可以用方向键连续前进、后退、左转、右转走完整张地图，并同屏实时观看**第三视角跟随小鸭子**、**机器人第一视角**和**俯视轨迹图**。

In [ ]:
# @title 1 · 安装原始 MicroDuck 仿真环境
project_source = "github"  # @param ["github", "drive", "upload"]
drive_folder = "microduck-qwen"  # @param {type:"string"}
render_backend = "osmesa"  # @param ["egl", "osmesa"]

import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
import zipfile
from pathlib import Path

if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("请选择 Python 3.12 或 3.13 的运行时，然后重新运行。")

root = Path("/content/microduck-qwen")
root.mkdir(parents=True, exist_ok=True)
bundle = "microduck-qwen-colab-source.zip"
local_zip = Path("/content") / bundle

if project_source == "github":
    with tempfile.TemporaryDirectory() as update:
        staged = Path(update) / "source"
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/atlasmarkz/microduck-qwen.git", str(staged)], check=True)
        subprocess.run(["git", "-C", str(staged), "sparse-checkout", "set", "microduck_qwen"], check=True)
        actual = subprocess.check_output(["git", "-C", str(staged), "rev-parse", "--short", "HEAD"], text=True).strip()
        shutil.copytree(staged / "microduck_qwen", root / "microduck_qwen", dirs_exist_ok=True)
        shutil.copyfile(staged / "pyproject.toml", root / "pyproject.toml")
        print("已从 GitHub 同步源码，提交：", actual)
elif project_source == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive") / drive_folder
    drive_zip = drive_dir / bundle
    if drive_zip.is_file():
        shutil.copyfile(drive_zip, local_zip)
        with zipfile.ZipFile(local_zip) as archive:
            archive.extractall(root)
        print("已从 Google Drive 解压最新源码包：", drive_zip)
    elif (drive_dir / "pyproject.toml").is_file() and (drive_dir / "microduck_qwen").is_dir():
        shutil.copytree(drive_dir / "microduck_qwen", root / "microduck_qwen", dirs_exist_ok=True)
        shutil.copyfile(drive_dir / "pyproject.toml", root / "pyproject.toml")
        print("已从 Google Drive 项目目录同步源码：", drive_dir)
    else:
        raise FileNotFoundError(f"未在 {drive_dir} 找到 {bundle} 或项目源码，也可切换到 github 或 upload。")
else:
    from google.colab import files
    uploaded = files.upload()
    if bundle not in uploaded:
        raise FileNotFoundError(f"请选择 {bundle}。")
    local_zip.write_bytes(uploaded[bundle])
    with zipfile.ZipFile(local_zip) as archive:
        archive.extractall(root)

os.environ["MUJOCO_GL"] = render_backend
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _mod in [m for m in list(sys.modules) if m == "microduck_qwen" or m.startswith("microduck_qwen.")]:
    del sys.modules[_mod]

def run(*cmd):
    completed = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        raise RuntimeError(completed.stdout[-8500:])

deps_marker = Path(f"/content/.microduck_deps_cpu_{render_backend}")
if deps_marker.is_file():
    print("当前会话依赖已安装，跳过重复安装。")
else:
    gl_packages = ("libegl1", "libgl1") if render_backend == "egl" else ("libosmesa6", "libgl1")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    run("apt-get", "-qq", "-y", "install", *gl_packages)
    project = tomllib.loads((root / "pyproject.toml").read_text())["project"]
    requirements = project["optional-dependencies"]
    bam = Path("/content/microduck-bam.zip")
    bam_url = next(item.split(" @ ", 1)[1] for item in requirements["simulation"] if " @ " in item)
    run("curl", "-fsSL", "--retry", "3", bam_url, "-o", str(bam))
    sim_packages = [item for item in requirements["simulation"] if " @ " not in item]
    with tempfile.TemporaryDirectory() as build:
        bam_source = bam
        if sys.version_info[:2] == (3, 13):
            with zipfile.ZipFile(bam) as archive:
                archive.extractall(build)
            manifest, = Path(build).glob("bam-*/pyproject.toml")
            config = manifest.read_text(encoding="utf-8")
            guard = 'requires-python = ">=3.12,<3.13"'
            if config.count(guard) != 1:
                raise RuntimeError("固定版 BAM 的 Python 声明与预期不符。")
            manifest.write_text(config.replace(guard, 'requires-python = ">=3.12,<3.14"'),
                                encoding="utf-8")
            bam_source = manifest.parent
        run(sys.executable, "-m", "pip", "install", "-q", "huggingface-hub", "Pillow>=10.1", "ipywidgets",
            *sim_packages, str(bam_source))
    deps_marker.write_text("ok")
from microduck_qwen.colab_demo import prepare_rendering
print("渲染器：", prepare_rendering(render_backend))
print("依赖安装完成。")

In [ ]:
# @title 2 · 下载官方 ONNX 步态和仿真资产
import tarfile
import tempfile
from pathlib import Path
from huggingface_hub import hf_hub_download
from microduck_qwen.sim_native import DEFAULT_POLICY, DEFAULT_RL, POLICY_REVISION
hf_hub_download("pollen-robotics/microduck-policies", "alpha_walking.onnx",
                revision=POLICY_REVISION, cache_dir=DEFAULT_POLICY.parents[3], token=False)

upstream_rev = "0daab160d3fb1c2cbaba16a4a19052f36709dd9e"
if not DEFAULT_RL.exists():
    DEFAULT_RL.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=DEFAULT_RL.parent) as temporary:
        upstream_archive = Path(temporary) / "upstream.tar.gz"
        run("curl", "-fsSL", "--retry", "3",
            f"https://codeload.github.com/pollen-robotics/microduck_rl/tar.gz/{upstream_rev}",
            "-o", str(upstream_archive))
        with tarfile.open(upstream_archive) as archive:
            archive.extractall(temporary, filter="data")
        (Path(temporary) / f"microduck_rl-{upstream_rev}").rename(DEFAULT_RL)

assert DEFAULT_POLICY.is_file(), "官方步态下载不完整。"
print("官方步态和仿真资产就绪。")

In [ ]:
# @title 3 · 启动交互式手动遥控台
seed = 98107  # @param {type:"integer"}

from microduck_qwen.colab_demo import interactive_baseline
console = interactive_baseline(seed=seed)